In [1]:
import torch
import gpytorch
import pandas as pd
import pickle
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from torch.utils.data import TensorDataset, DataLoader
from pyproj import Transformer
from sklearn.metrics import pairwise_distances
from scipy.interpolate import RegularGridInterpolator
from torch_geometric.data import Data




/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd

air_korea_final = pd.read_pickle("/Users/drewbaldwin/PM2_5 Research/air_korea_final_imputed_with_blh.pkl")
air_korea_final.head()


,Datetime,SO2,CO,O3,NO2,PM10,PM25,Station_ID,Year,lon,...,urban_landuse_area_m2_3km,green_space_area_3km,building_footprint_area_3km,railway_length_3km,dist_to_coast_km,dist_to_major_road_km,industrial_area_m2_3km,traffic_points_count_3km,boundary_layer_height,cloud_cover
0,2016-01-01 00:00:00,7.0,1000.0,2.0,76.0,77.0,53.0,111121,2016,126.9747,...,2.952433e+06,9.417996e+06,4.942823e+06,114737.280122,10.830496,0.102038,1489.332477,159.0,30.0,6
1,2016-01-01 01:00:00,7.0,1100.0,2.0,77.0,70.0,48.0,111121,2016,126.9747,...,2.952433e+06,9.417996e+06,4.942823e+06,114737.280122,10.830496,0.102038,1489.332477,159.0,25.0,3
2,2016-01-01 02:00:00,7.0,1200.0,2.0,78.0,75.0,53.0,111121,2016,126.9747,...,2.952433e+06,9.417996e+06,4.942823e+06,114737.280122,10.830496,0.102038,1489.332477,159.0,15.0,2
3,2016-01-01 03:00:00,6.0,1400.0,2.0,78.0,77.0,53.0,111121,2016,126.9747,...,2.952433e+06,9.417996e+06,4.942823e+06,114737.280122,10.830496,0.102038,1489.332477,159.0,15.0,1
4,2016-01-01 04:00:00,6.0,1500.0,2.0,77.0,83.0,52.0,111121,2016,126.9747,...,2.952433e+06,9.417996e+06,4.942823e+06,114737.280122,10.830496,0.102038,1489.332477,159.0,15.0,1


In [6]:
# ================================================================
# 1) Temporal correlation: autocorrelation of PM2.5 at each station,
# both on the RAW series (includes diurnal/seasonal cycle -- shows
# full predictive utility of self-history) and on the DESEASONALIZED
# anomaly (cross-station mean removed -- shows genuine local
# persistence beyond the shared national signal).
# ================================================================
import numpy as np
import pandas as pd

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())

pm25_wide = df.pivot(index="Datetime", columns="Station_ID", values="PM25")[station_order]
national_mean = pm25_wide.mean(axis=1)
resid = pm25_wide.sub(national_mean, axis=0)

LAGS = [1, 2, 3, 6, 12, 24, 48, 72, 168]  # hours, out to 1 week

def acf_per_station(wide_df, lags):
    out = {}
    for lag in lags:
        corrs = [wide_df[col].autocorr(lag=lag) for col in wide_df.columns]
        out[lag] = (np.nanmean(corrs), np.nanstd(corrs))
    return out

print("Autocorrelation of RAW PM2.5 (includes diurnal/seasonal cycle):")
for lag, (m, s) in acf_per_station(pm25_wide, LAGS).items():
    print(f"  lag={lag:4d}h: mean={m:.3f}, std across stations={s:.3f}")

print("\nAutocorrelation of DESEASONALIZED PM2.5 anomaly (genuine local persistence, "
      "not just the shared national/diurnal pattern):")
for lag, (m, s) in acf_per_station(resid, LAGS).items():
    print(f"  lag={lag:4d}h: mean={m:.3f}, std across stations={s:.3f}")


Autocorrelation of RAW PM2.5 (includes diurnal/seasonal cycle):
  lag=   1h: mean=0.948, std across stations=0.017
  lag=   2h: mean=0.900, std across stations=0.026
  lag=   3h: mean=0.855, std across stations=0.032
  lag=   6h: mean=0.748, std across stations=0.041
  lag=  12h: mean=0.614, std across stations=0.045
  lag=  24h: mean=0.479, std across stations=0.041
  lag=  48h: mean=0.288, std across stations=0.044
  lag=  72h: mean=0.229, std across stations=0.046
  lag= 168h: mean=0.198, std across stations=0.052

Autocorrelation of DESEASONALIZED PM2.5 anomaly (genuine local persistence, not just the shared national/diurnal pattern):
  lag=   1h: mean=0.881, std across stations=0.030
  lag=   2h: mean=0.779, std across stations=0.046
  lag=   3h: mean=0.694, std across stations=0.059
  lag=   6h: mean=0.517, std across stations=0.074
  lag=  12h: mean=0.349, std across stations=0.074
  lag=  24h: mean=0.297, std across stations=0.050
  lag=  48h: mean=0.200, std across stations=0.

In [7]:
# ================================================================
# 2) Every other variable in the dataset vs PM2.5: pooled correlation
# (raw and deseasonalized) for time-varying predictors, and
# correlation against per-station mean PM2.5 for static covariates.
# Flags expected near-circular relationships (PM10) rather than
# treating them as a genuine finding.
# ================================================================
TIME_VARYING_COLS = ["SO2", "CO", "O3", "NO2", "PM10",
                      "windspeed_10m", "surface_pressure", "temperature_2m",
                      "relative_humidity_2m", "precipitation",
                      "boundary_layer_height", "cloud_cover"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

def pooled_corr(a_wide, b_wide):
    a, b = a_wide.to_numpy().ravel(), b_wide.to_numpy().ravel()
    valid = ~(np.isnan(a) | np.isnan(b))
    return np.corrcoef(a[valid], b[valid])[0, 1] if valid.sum() > 1 else np.nan

print("=== Time-varying predictors vs PM2.5 (pooled across all station-hours) ===")
results = []
for col in TIME_VARYING_COLS:
    wide_var = df.pivot(index="Datetime", columns="Station_ID", values=col)[station_order]
    raw_corr = pooled_corr(pm25_wide, wide_var)
    resid_var = wide_var.sub(wide_var.mean(axis=1), axis=0)
    deseason_corr = pooled_corr(resid, resid_var)
    flag = "  <-- PM10 physically contains PM2.5, expect near-circular correlation" if col == "PM10" else ""
    results.append({"variable": col, "raw_corr": raw_corr, "deseasonalized_corr": deseason_corr, "note": flag})

# wind direction is circular -- plain correlation isn't meaningful, use sin/cos components
wdir_wide = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order]
wdir_rad = np.radians(wdir_wide)
for label, comp in [("winddirection_10m (sin)", np.sin(wdir_rad)), ("winddirection_10m (cos)", np.cos(wdir_rad))]:
    raw_corr = pooled_corr(pm25_wide, comp)
    comp_resid = comp.sub(comp.mean(axis=1), axis=0)
    deseason_corr = pooled_corr(resid, comp_resid)
    results.append({"variable": label, "raw_corr": raw_corr, "deseasonalized_corr": deseason_corr, "note": ""})

results_df = pd.DataFrame(results).sort_values("deseasonalized_corr", key=abs, ascending=False)
print(results_df.to_string(index=False))

print("\n=== Static covariates vs per-station mean PM2.5 ===")
station_mean_pm25 = pm25_wide.mean(axis=0)
static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]

static_results = []
for col in STATIC_COLS:
    c = np.corrcoef(static_df[col].to_numpy(), station_mean_pm25.to_numpy())[0, 1]
    static_results.append({"variable": col, "corr_with_station_mean_PM25": c})

static_results_df = pd.DataFrame(static_results).sort_values("corr_with_station_mean_PM25", key=abs, ascending=False)
print(static_results_df.to_string(index=False))


=== Time-varying predictors vs PM2.5 (pooled across all station-hours) ===
               variable  raw_corr  deseasonalized_corr                                                                   note
                   PM10  0.757078             0.698141   <-- PM10 physically contains PM2.5, expect near-circular correlation
                     CO  0.519613             0.307338                                                                       
                    NO2  0.444787             0.267554                                                                       
                    SO2  0.267276             0.149887                                                                       
         temperature_2m -0.201745            -0.097103                                                                       
  boundary_layer_height -0.204373            -0.092809                                                                       
          windspeed_10m -0.214496          

In [2]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
import copy

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())

STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]
POLLUTANT_COLS = ["SO2", "CO", "NO2", "O3", "PM10"]
WEATHER_COLS = ["windspeed_10m", "temperature_2m", "relative_humidity_2m",
                "precipitation", "boundary_layer_height", "cloud_cover", "surface_pressure"]
H = 24  # forecast horizon, hours

def build_station_features(g):
    g = g.sort_values("Datetime").set_index("Datetime")
    out = pd.DataFrame(index=g.index)

    for lag in [0, 1, 3, 6, 12, 24, 48, 72]:
        out[f"pm25_lag{lag}"] = g["PM25"].shift(lag)
    for win in [6, 24, 72]:
        r = g["PM25"].rolling(win, min_periods=max(1, win // 3))
        out[f"pm25_roll_mean{win}"] = r.mean()
        out[f"pm25_roll_std{win}"] = r.std()
        out[f"pm25_roll_max{win}"] = r.max()

    for c in POLLUTANT_COLS:
        out[c] = g[c]
        out[f"{c}_roll_mean24"] = g[c].rolling(24, min_periods=8).mean()

    wdir_rad = np.radians(g["winddirection_10m"])
    out["wdir_sin"], out["wdir_cos"] = np.sin(wdir_rad), np.cos(wdir_rad)
    for c in WEATHER_COLS:
        out[c] = g[c]
    for c in ["windspeed_10m", "boundary_layer_height", "temperature_2m"]:
        out[f"{c}_roll_mean24"] = g[c].rolling(24, min_periods=8).mean()

    hour = g.index.hour
    doy = g.index.dayofyear
    out["hour_sin"], out["hour_cos"] = np.sin(2*np.pi*hour/24), np.cos(2*np.pi*hour/24)
    out["doy_sin"], out["doy_cos"] = np.sin(2*np.pi*doy/365.25), np.cos(2*np.pi*doy/365.25)

    for c in STATIC_COLS:
        out[c] = g[c].iloc[0]

    out["target"] = g["PM25"].shift(-H)
    out["Station_ID"] = g["Station_ID"].iloc[0]
    return out

print("Engineering features per station (this takes a few minutes)...")
feat_list = [build_station_features(g) for _, g in df.groupby("Station_ID")]
feat_df = pd.concat(feat_list).dropna()
feat_df["Year"] = feat_df.index.year
print(f"feature matrix: {feat_df.shape[0]:,} rows x {feat_df.shape[1]} cols")

feature_cols = [c for c in feat_df.columns if c not in ("target", "Station_ID", "Year")]
train = feat_df[feat_df["Year"] <= 2018]
val = feat_df[feat_df["Year"] == 2019]
test = feat_df[feat_df["Year"] >= 2020]
print(f"train={len(train):,}  val={len(val):,}  test={len(test):,}")

X_train_arr, y_train_arr = train[feature_cols].to_numpy(), train["target"].to_numpy()
X_val_arr, y_val_arr = val[feature_cols].to_numpy(), val["target"].to_numpy()
X_test_arr, y_test_arr = test[feature_cols].to_numpy(), test["target"].to_numpy()

model_G = HistGradientBoostingRegressor(
    max_depth=6, learning_rate=0.05, max_iter=0, warm_start=True,
    l2_regularization=1.0, random_state=0
)

CHUNK, PATIENCE = 25, 4
best_val_rmse, best_iter, no_improve, best_model = float("inf"), 0, 0, None
for chunk in range(1, 81):  # up to 2000 rounds
    model_G.max_iter = chunk * CHUNK
    model_G.fit(X_train_arr, y_train_arr)
    val_pred = model_G.predict(X_val_arr)
    val_rmse = np.sqrt(np.mean((val_pred - y_val_arr) ** 2))
    print(f"n_iter={chunk*CHUNK:4d}  val_rmse={val_rmse:.4f}")
    if val_rmse < best_val_rmse - 1e-4:
        best_val_rmse, best_iter, no_improve = val_rmse, chunk * CHUNK, 0
        best_model = copy.deepcopy(model_G)
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping: best n_iter={best_iter}, val_rmse={best_val_rmse:.4f}")
            break

model_G = best_model

for name, X, y in [("train", X_train_arr, y_train_arr), ("val", X_val_arr, y_val_arr), ("test", X_test_arr, y_test_arr)]:
    pred = model_G.predict(X)
    rmse = np.sqrt(np.mean((pred - y) ** 2))
    mae = np.mean(np.abs(pred - y))
    print(f"{name}: RMSE={rmse:.3f}  MAE={mae:.3f}")

rng = np.random.default_rng(0)
sub_idx = rng.choice(len(X_val_arr), size=min(20000, len(X_val_arr)), replace=False)
perm = permutation_importance(model_G, X_val_arr[sub_idx], y_val_arr[sub_idx],
                               n_repeats=5, random_state=0, n_jobs=-1)
importance = pd.Series(perm.importances_mean, index=feature_cols).sort_values(ascending=False)
print("\nTop 15 features by permutation importance:")
print(importance.head(15).to_string())


Engineering features per station (this takes a few minutes)...
feature matrix: 9,242,112 rows x 57 cols
train=4,616,832  val=1,541,760  test=3,083,520
n_iter=  25  val_rmse=16.0308
n_iter=  50  val_rmse=15.4862
n_iter=  75  val_rmse=15.3618
n_iter= 100  val_rmse=15.3357
n_iter= 125  val_rmse=15.3914
n_iter= 150  val_rmse=15.4237
n_iter= 175  val_rmse=15.4918
n_iter= 200  val_rmse=15.5344
Early stopping: best n_iter=100, val_rmse=15.3357
train: RMSE=12.991  MAE=9.508
val: RMSE=15.336  MAE=10.415
test: RMSE=12.627  MAE=9.107

Top 15 features by permutation importance:
pm25_lag0                     0.170464
temperature_2m_roll_mean24    0.027060
windspeed_10m                 0.021642
pm25_lag72                    0.013703
pm25_lag1                     0.009004
pm25_roll_mean72              0.008855
surface_pressure              0.008035
boundary_layer_height         0.007927
doy_sin                       0.007925
cloud_cover                   0.007692
wdir_sin                      0.00620

In [3]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

pred_train = model_G.predict(X_train_arr)
pred_val = model_G.predict(X_val_arr)
pred_test = model_G.predict(X_test_arr)

vol_col = feature_cols.index("pm25_roll_std24")
vol_train, vol_val, vol_test = X_train_arr[:, vol_col], X_val_arr[:, vol_col], X_test_arr[:, vol_col]

THRESHOLDS = {"bad_36": 36.0, "very_bad_76": 76.0}
calibrators = {}

for name, c in THRESHOLDS.items():
    y_train_label = (y_train_arr > c).astype(int)
    y_val_label = (y_val_arr > c).astype(int)
    y_test_label = (y_test_arr > c).astype(int)

    print(f"\n=== threshold {name} (c={c}) ===")
    print(f"base rate: train={y_train_label.mean():.4f}  val={y_val_label.mean():.4f}  test={y_test_label.mean():.4f}")

    Zval = np.column_stack([pred_val, vol_val])
    S = LogisticRegression(max_iter=1000)
    S.fit(Zval, y_val_label)
    calibrators[name] = S

    Ztest = np.column_stack([pred_test, vol_test])
    p_test = S.predict_proba(Ztest)[:, 1]

    auc = roc_auc_score(y_test_label, p_test)
    pr_auc = average_precision_score(y_test_label, p_test)
    brier = brier_score_loss(y_test_label, p_test)
    print(f"test: AUC={auc:.4f}  PR-AUC={pr_auc:.4f}  Brier={brier:.4f}")

    # reliability check: bin predicted probability, compare to observed frequency
    bins = np.quantile(p_test, np.linspace(0, 1, 11))
    bins[-1] += 1e-6
    bin_idx = np.digitize(p_test, bins) - 1
    print("reliability (predicted vs observed, by decile of predicted prob):")
    for b in range(10):
        mask = bin_idx == b
        if mask.sum() > 0:
            print(f"  decile {b}: n={mask.sum():6d}  mean_pred={p_test[mask].mean():.4f}  observed_rate={y_test_label[mask].mean():.4f}")



=== threshold bad_36 (c=36.0) ===
base rate: train=0.1910  val=0.1553  test=0.0947
test: AUC=0.8427  PR-AUC=0.3869  Brier=0.0707
reliability (predicted vs observed, by decile of predicted prob):
  decile 0: n=308352  mean_pred=0.0172  observed_rate=0.0018
  decile 1: n=308352  mean_pred=0.0271  observed_rate=0.0049
  decile 2: n=308352  mean_pred=0.0371  observed_rate=0.0107
  decile 3: n=308352  mean_pred=0.0483  observed_rate=0.0207
  decile 4: n=308352  mean_pred=0.0618  observed_rate=0.0377
  decile 5: n=308352  mean_pred=0.0793  observed_rate=0.0593
  decile 6: n=308352  mean_pred=0.1032  observed_rate=0.0857
  decile 7: n=308352  mean_pred=0.1403  observed_rate=0.1225
  decile 8: n=308352  mean_pred=0.2127  observed_rate=0.1970
  decile 9: n=308352  mean_pred=0.4591  observed_rate=0.4066

=== threshold very_bad_76 (c=76.0) ===
base rate: train=0.0137  val=0.0218  test=0.0070
test: AUC=0.8376  PR-AUC=0.0508  Brier=0.0075
reliability (predicted vs observed, by decile of predicted 

In [4]:
national_daily = df.pivot(index="Datetime", columns="Station_ID", values="PM25").resample("D").mean()

TRAIL_DAYS = 90
trailing_rate = {}
for name, c in THRESHOLDS.items():
    frac_exceeding = (national_daily > c).mean(axis=1)
    trailing_rate[name] = frac_exceeding.rolling(TRAIL_DAYS, min_periods=30).mean().shift(1)  # past-only

feat_dates = feat_df.index.normalize()
for name in THRESHOLDS:
    feat_df[f"trail_rate_{name}"] = trailing_rate[name].reindex(feat_dates).to_numpy()

train = feat_df[feat_df["Year"] <= 2018]
val = feat_df[feat_df["Year"] == 2019]
test = feat_df[feat_df["Year"] >= 2020]

for name, c in THRESHOLDS.items():
    y_val_label = (val["target"] > c).astype(int).to_numpy()
    y_test_label = (test["target"] > c).astype(int).to_numpy()
    trail_val = np.nan_to_num(val[f"trail_rate_{name}"].to_numpy(), nan=frac_exceeding.mean())
    trail_test = np.nan_to_num(test[f"trail_rate_{name}"].to_numpy(), nan=frac_exceeding.mean())

    Zval = np.column_stack([pred_val, vol_val, trail_val])
    S = LogisticRegression(max_iter=1000)
    S.fit(Zval, y_val_label)
    calibrators[name] = S

    Ztest = np.column_stack([pred_test, vol_test, trail_test])
    p_test = S.predict_proba(Ztest)[:, 1]

    auc = roc_auc_score(y_test_label, p_test)
    pr_auc = average_precision_score(y_test_label, p_test)
    brier = brier_score_loss(y_test_label, p_test)
    trivial = y_test_label.mean() * (1 - y_test_label.mean())
    print(f"\n=== {name} (c={c}) ===")
    print(f"test: AUC={auc:.4f}  PR-AUC={pr_auc:.4f}  Brier={brier:.4f}  (trivial={trivial:.4f})")

    bins = np.quantile(p_test, np.linspace(0, 1, 11)); bins[-1] += 1e-6
    bin_idx = np.digitize(p_test, bins) - 1
    for b in range(10):
        mask = bin_idx == b
        if mask.sum() > 0:
            print(f"  decile {b}: n={mask.sum():6d}  mean_pred={p_test[mask].mean():.4f}  observed_rate={y_test_label[mask].mean():.4f}")



=== bad_36 (c=36.0) ===
test: AUC=0.8426  PR-AUC=0.3849  Brier=0.0704  (trivial=0.0857)
  decile 0: n=308352  mean_pred=0.0154  observed_rate=0.0018
  decile 1: n=308352  mean_pred=0.0242  observed_rate=0.0047
  decile 2: n=308352  mean_pred=0.0337  observed_rate=0.0100
  decile 3: n=308352  mean_pred=0.0445  observed_rate=0.0202
  decile 4: n=308352  mean_pred=0.0572  observed_rate=0.0369
  decile 5: n=308352  mean_pred=0.0734  observed_rate=0.0608
  decile 6: n=308352  mean_pred=0.0954  observed_rate=0.0878
  decile 7: n=308352  mean_pred=0.1294  observed_rate=0.1234
  decile 8: n=308352  mean_pred=0.1960  observed_rate=0.1966
  decile 9: n=308352  mean_pred=0.4284  observed_rate=0.4047

=== very_bad_76 (c=76.0) ===
test: AUC=0.8408  PR-AUC=0.0502  Brier=0.0073  (trivial=0.0070)
  decile 0: n=308352  mean_pred=0.0005  observed_rate=0.0000
  decile 1: n=308352  mean_pred=0.0009  observed_rate=0.0000
  decile 2: n=308352  mean_pred=0.0013  observed_rate=0.0002
  decile 3: n=308352  me

In [6]:
resid_national_mean = resid_wide.mean(axis=1)
resid_local = resid_wide.sub(resid_national_mean, axis=0)

total_var = resid_wide.var().mean()
national_var = resid_national_mean.var()
print(f"national common-factor variance: {national_var:.3f} of mean per-station residual variance {total_var:.3f} "
      f"({national_var/total_var:.1%})")

resid_local_corr = resid_local.corr().to_numpy()[iu]
print("\nLOCAL (national-mean-removed) residual correlation by distance:")
for lo, hi in zip(bins[:-1], bins[1:]):
    mask = (dist_flat >= lo) & (dist_flat < hi)
    if mask.sum() > 0:
        print(f"{lo:3d}-{hi:3d}km: n_pairs={mask.sum():5d}  mean_local_residual_corr={resid_local_corr[mask].mean():.4f}")


national common-factor variance: 77.005 of mean per-station residual variance 149.662 (51.5%)

LOCAL (national-mean-removed) residual correlation by distance:
  0- 25km: n_pairs= 1323  mean_local_residual_corr=0.6090
 25- 50km: n_pairs= 1375  mean_local_residual_corr=0.4038
 50- 75km: n_pairs=  752  mean_local_residual_corr=0.3032
 75-100km: n_pairs= 1098  mean_local_residual_corr=0.1876
100-150km: n_pairs= 2125  mean_local_residual_corr=0.0543
150-200km: n_pairs= 1834  mean_local_residual_corr=-0.0447
200-300km: n_pairs= 4243  mean_local_residual_corr=-0.2257
300-500km: n_pairs= 2633  mean_local_residual_corr=-0.3618


In [8]:
LAGS_TO_TEST = [1, 6, 24]
n = len(station_cols)
mask_offdiag = ~np.eye(n, dtype=bool)
dist_flat_all = dist_km[mask_offdiag]

for LAG in LAGS_TO_TEST:
    resid_local_lag = resid_local.shift(LAG)
    combined = pd.concat([resid_local.add_suffix("_now"), resid_local_lag.add_suffix("_lag")], axis=1)
    full_corr = combined.corr()
    now_cols = [f"{c}_now" for c in station_cols]
    lag_cols = [f"{c}_lag" for c in station_cols]
    cross_corr = full_corr.loc[now_cols, lag_cols].to_numpy()
    cross_corr_flat = cross_corr[mask_offdiag]

    print(f"\n=== LAG={LAG}h: does station B's ALREADY-KNOWN residual from {LAG}h ago predict station A's current one? ===")
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (dist_flat_all >= lo) & (dist_flat_all < hi)
        if m.sum() > 0:
            print(f"{lo:3d}-{hi:3d}km: n_pairs={m.sum():5d}  mean_lagged_local_corr={cross_corr_flat[m].mean():.4f}")



=== LAG=1h: does station B's ALREADY-KNOWN residual from 1h ago predict station A's current one? ===
  0- 25km: n_pairs= 2646  mean_lagged_local_corr=0.5860
 25- 50km: n_pairs= 2750  mean_lagged_local_corr=0.3968
 50- 75km: n_pairs= 1504  mean_lagged_local_corr=0.2991
 75-100km: n_pairs= 2196  mean_lagged_local_corr=0.1874
100-150km: n_pairs= 4250  mean_lagged_local_corr=0.0566
150-200km: n_pairs= 3668  mean_lagged_local_corr=-0.0416
200-300km: n_pairs= 8486  mean_lagged_local_corr=-0.2198
300-500km: n_pairs= 5266  mean_lagged_local_corr=-0.3542

=== LAG=6h: does station B's ALREADY-KNOWN residual from 6h ago predict station A's current one? ===
  0- 25km: n_pairs= 2646  mean_lagged_local_corr=0.3423
 25- 50km: n_pairs= 2750  mean_lagged_local_corr=0.2614
 50- 75km: n_pairs= 1504  mean_lagged_local_corr=0.1983
 75-100km: n_pairs= 2196  mean_lagged_local_corr=0.1378
100-150km: n_pairs= 4250  mean_lagged_local_corr=0.0519
150-200km: n_pairs= 3668  mean_lagged_local_corr=-0.0176
200-300k

In [9]:
fine_bins = [0, 5, 10, 15, 20, 25, 35, 50, 75, 100]
resid_local_lag24 = resid_local.shift(24)
combined = pd.concat([resid_local.add_suffix("_now"), resid_local_lag24.add_suffix("_lag")], axis=1)
full_corr = combined.corr()
now_cols = [f"{c}_now" for c in station_cols]
lag_cols = [f"{c}_lag" for c in station_cols]
cross_corr = full_corr.loc[now_cols, lag_cols].to_numpy()
cross_corr_flat = cross_corr[mask_offdiag]

print("Fine-grained LAG=24h local residual correlation by distance:")
for lo, hi in zip(fine_bins[:-1], fine_bins[1:]):
    m = (dist_flat_all >= lo) & (dist_flat_all < hi)
    if m.sum() > 0:
        print(f"{lo:3d}-{hi:3d}km: n_pairs={m.sum():5d}  mean_lagged_local_corr={cross_corr_flat[m].mean():.4f}")

print(f"\nexisting pairs closer than 10km: {(dist_flat_all < 10).sum()}")
print(f"existing pairs closer than 5km: {(dist_flat_all < 5).sum()}")


Fine-grained LAG=24h local residual correlation by distance:
  0-  5km: n_pairs=  186  mean_lagged_local_corr=-0.0644
  5- 10km: n_pairs=  526  mean_lagged_local_corr=-0.0518
 10- 15km: n_pairs=  638  mean_lagged_local_corr=-0.0388
 15- 20km: n_pairs=  644  mean_lagged_local_corr=-0.0258
 20- 25km: n_pairs=  652  mean_lagged_local_corr=-0.0182
 25- 35km: n_pairs= 1346  mean_lagged_local_corr=-0.0117
 35- 50km: n_pairs= 1404  mean_lagged_local_corr=-0.0085
 50- 75km: n_pairs= 1504  mean_lagged_local_corr=-0.0064
 75-100km: n_pairs= 2196  mean_lagged_local_corr=0.0046

existing pairs closer than 10km: 712
existing pairs closer than 5km: 186


In [10]:
from sklearn.feature_selection import mutual_info_regression

K_NEIGHBORS = 5
n_stations = len(station_cols)

resid_arr = resid_wide[station_cols].to_numpy()  # [T, N], from the residual-decomposition work above
resid_arr = np.nan_to_num(resid_arr, nan=0.0)

# subsample timesteps -- KNN-based MI scales poorly with number of observations
rng = np.random.default_rng(0)
SAMPLE_HOURS = min(8760, resid_arr.shape[0])
sample_idx = rng.choice(resid_arr.shape[0], size=SAMPLE_HOURS, replace=False)
resid_sample = resid_arr[sample_idx]

neighbor_mi = np.zeros(n_stations)
for i in range(n_stations):
    neighbor_order = np.argsort(dist_km[i])
    neighbor_order = neighbor_order[neighbor_order != i][:K_NEIGHBORS]
    X_neighbors = resid_sample[:, neighbor_order]
    y_target = resid_sample[:, i]
    mi = mutual_info_regression(X_neighbors, y_target, random_state=0, n_neighbors=3)
    neighbor_mi[i] = mi.sum()  # total MI a station's K nearest neighbors carry about its residual

# cross-reference with actual forecast error, for policy context: low neighbor-MI + high
# error together identify locations that are both under-covered AND hard to predict
test_meta2 = test[["Station_ID"]].copy()
test_meta2["abs_err"] = np.abs(y_test_arr - pred_test)
station_mae = test_meta2.groupby("Station_ID")["abs_err"].mean()

stations_meta = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID")

mi_df = pd.DataFrame({
    "Station_ID": station_cols,
    "neighbor_mi": neighbor_mi,
})
mi_df["station_mae"] = mi_df["Station_ID"].map(station_mae)
mi_df["lat"] = mi_df["Station_ID"].map(stations_meta["lat"])
mi_df["lon"] = mi_df["Station_ID"].map(stations_meta["lon"])
mi_df = mi_df.sort_values("neighbor_mi")

print(f"=== Stations LEAST explained by their {K_NEIGHBORS} nearest neighbors' residuals ===")
print("(low MI = under-covered by existing network = best candidates for added sensor density)")
print(mi_df.head(15).to_string(index=False))

print(f"\n=== Stations BEST explained by their {K_NEIGHBORS} nearest neighbors' residuals ===")
print("(high MI = redundant with existing coverage -- a new sensor here would add little)")
print(mi_df.tail(15).to_string(index=False))

print(f"\noverall neighbor MI: mean={mi_df['neighbor_mi'].mean():.4f}  "
      f"std={mi_df['neighbor_mi'].std():.4f}  "
      f"min={mi_df['neighbor_mi'].min():.4f}  max={mi_df['neighbor_mi'].max():.4f}")

corr_mi_err = mi_df["neighbor_mi"].corr(mi_df["station_mae"])
print(f"correlation between neighbor-MI and forecast error (MAE): {corr_mi_err:.4f}")


=== Stations LEAST explained by their 5 nearest neighbors' residuals ===
(low MI = under-covered by existing network = best candidates for added sensor density)
 Station_ID  neighbor_mi  station_mae     lat      lon
     339121     0.934726     7.999667 33.2465 126.5653
     339112     1.503031     7.706540 33.4886 126.5001
     339111     1.563688     8.339622 33.5001 126.5324
     632132     1.694378     8.231334 37.7600 128.9030
     534422     1.695458    10.365871 36.7799 126.4561
     632151     1.778082     7.731173 37.5250 129.1145
     238201     1.802877     8.257940 34.8660 128.6922
     336441     1.862646     9.272467 34.7668 126.4392
     632161     1.877826     8.296339 37.4426 129.1684
     735151     1.906510     8.560503 35.4077 127.3846
     437112     1.914263     7.510373 35.9802 129.3748
     336121     1.944097     7.652614 34.7411 127.7260
     132112     1.968265     9.533627 37.8756 127.7205
     336112     1.970114     8.793414 34.8043 126.4346
     525161   

In [13]:
H6 = 6
pm25_shifted = df[["Datetime", "Station_ID", "PM25"]].sort_values(["Station_ID", "Datetime"]).copy()
pm25_shifted["target_h6"] = pm25_shifted.groupby("Station_ID")["PM25"].shift(-H6)

feat_df_h6 = feat_df.reset_index().rename(columns={feat_df.index.name or "index": "Datetime"})
feat_df_h6 = feat_df_h6.merge(pm25_shifted[["Datetime", "Station_ID", "target_h6"]], on=["Datetime", "Station_ID"], how="left")
feat_df_h6 = feat_df_h6.drop(columns=["target"]).rename(columns={"target_h6": "target"})
feat_df_h6 = feat_df_h6.dropna(subset=["target"]).set_index("Datetime")
print(f"H=6 feature matrix: {feat_df_h6.shape[0]:,} rows")

train6 = feat_df_h6[feat_df_h6["Year"] <= 2018]
val6 = feat_df_h6[feat_df_h6["Year"] == 2019]
test6 = feat_df_h6[feat_df_h6["Year"] >= 2020]

X_train6, y_train6 = train6[feature_cols].to_numpy(), train6["target"].to_numpy()
X_val6, y_val6 = val6[feature_cols].to_numpy(), val6["target"].to_numpy()
X_test6, y_test6 = test6[feature_cols].to_numpy(), test6["target"].to_numpy()

model_G6 = HistGradientBoostingRegressor(max_depth=6, learning_rate=0.05, max_iter=0, warm_start=True,
                                          l2_regularization=1.0, random_state=0)
CHUNK, PATIENCE = 25, 4
best_val_rmse, best_iter, no_improve, best_model = float("inf"), 0, 0, None
for chunk in range(1, 81):
    model_G6.max_iter = chunk * CHUNK
    model_G6.fit(X_train6, y_train6)
    val_pred = model_G6.predict(X_val6)
    val_rmse = np.sqrt(np.mean((val_pred - y_val6) ** 2))
    print(f"n_iter={chunk*CHUNK:4d}  val_rmse={val_rmse:.4f}")
    if val_rmse < best_val_rmse - 1e-4:
        best_val_rmse, best_iter, no_improve = val_rmse, chunk * CHUNK, 0
        best_model = copy.deepcopy(model_G6)
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping: best n_iter={best_iter}, val_rmse={best_val_rmse:.4f}")
            break
model_G6 = best_model

pred_train6, pred_val6, pred_test6 = model_G6.predict(X_train6), model_G6.predict(X_val6), model_G6.predict(X_test6)
for name, y, pred in [("train", y_train6, pred_train6), ("val", y_val6, pred_val6), ("test", y_test6, pred_test6)]:
    rmse = np.sqrt(np.mean((pred - y) ** 2))
    print(f"{name}: RMSE={rmse:.3f}")


H=6 feature matrix: 9,242,112 rows
n_iter=  25  val_rmse=12.4580
n_iter=  50  val_rmse=11.3133
n_iter=  75  val_rmse=10.9561
n_iter= 100  val_rmse=10.8267
n_iter= 125  val_rmse=10.7580
n_iter= 150  val_rmse=10.7129
n_iter= 175  val_rmse=10.6673
n_iter= 200  val_rmse=10.6566
n_iter= 225  val_rmse=10.6425
n_iter= 250  val_rmse=10.6261
n_iter= 275  val_rmse=10.6143
n_iter= 300  val_rmse=10.6122
n_iter= 325  val_rmse=10.6062
n_iter= 350  val_rmse=10.6078
n_iter= 375  val_rmse=10.6038
n_iter= 400  val_rmse=10.6062
n_iter= 425  val_rmse=10.6137
n_iter= 450  val_rmse=10.6120
n_iter= 475  val_rmse=10.6125
Early stopping: best n_iter=375, val_rmse=10.6038
train: RMSE=9.515
val: RMSE=10.604
test: RMSE=8.993


In [14]:
resid_test6 = y_test6 - pred_test6
test_meta6 = test6[["Station_ID"]].copy()
test_meta6["resid"] = resid_test6
resid_wide6 = test_meta6.pivot_table(index=test_meta6.index, columns="Station_ID", values="resid")

resid_national_mean6 = resid_wide6.mean(axis=1)
resid_local6 = resid_wide6.sub(resid_national_mean6, axis=0).reindex(columns=station_cols)

total_var6 = resid_wide6.var().mean()
national_var6 = resid_national_mean6.var()
print(f"H=6: national common-factor variance: {national_var6:.3f} of {total_var6:.3f} ({national_var6/total_var6:.1%})")

LAG6 = 6  # the boundary of legitimate availability for an H=6 model
resid_local6_lag = resid_local6.shift(LAG6)
combined6 = pd.concat([resid_local6.add_suffix("_now"), resid_local6_lag.add_suffix("_lag")], axis=1)
full_corr6 = combined6.corr()
now_cols6 = [f"{c}_now" for c in station_cols]
lag_cols6 = [f"{c}_lag" for c in station_cols]
cross_corr6_flat = full_corr6.loc[now_cols6, lag_cols6].to_numpy()[mask_offdiag]

print(f"\nH=6 model: LAG={LAG6}h local residual correlation by distance (legitimately available at prediction time):")
for lo, hi in zip(fine_bins[:-1], fine_bins[1:]):
    m = (dist_flat_all >= lo) & (dist_flat_all < hi)
    if m.sum() > 0:
        print(f"{lo:3d}-{hi:3d}km: n_pairs={m.sum():5d}  mean_lagged_local_corr={cross_corr6_flat[m].mean():.4f}")


H=6: national common-factor variance: 23.223 of 78.113 (29.7%)

H=6 model: LAG=6h local residual correlation by distance (legitimately available at prediction time):
  0-  5km: n_pairs=  186  mean_lagged_local_corr=-0.0109
  5- 10km: n_pairs=  526  mean_lagged_local_corr=0.0133
 10- 15km: n_pairs=  638  mean_lagged_local_corr=0.0290
 15- 20km: n_pairs=  644  mean_lagged_local_corr=0.0421
 20- 25km: n_pairs=  652  mean_lagged_local_corr=0.0456
 25- 35km: n_pairs= 1346  mean_lagged_local_corr=0.0483
 35- 50km: n_pairs= 1404  mean_lagged_local_corr=0.0505
 50- 75km: n_pairs= 1504  mean_lagged_local_corr=0.0394
 75-100km: n_pairs= 2196  mean_lagged_local_corr=0.0398


In [15]:
SEOUL_STATIONS = [sid for sid in station_cols if str(sid).startswith("111")]
print(f"{len(SEOUL_STATIONS)} Seoul stations: {SEOUL_STATIONS}")

seoul_resid = resid_local[SEOUL_STATIONS].mean(axis=1)

seoul_lat = stations_meta.loc[SEOUL_STATIONS, "lat"].mean()
seoul_lon = stations_meta.loc[SEOUL_STATIONS, "lon"].mean()

def bearing_deg(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

other_stations2 = [s for s in station_cols if s not in SEOUL_STATIONS]
bearings = {s: bearing_deg(seoul_lat, seoul_lon, stations_meta.loc[s, "lat"], stations_meta.loc[s, "lon"]) for s in other_stations2}

wind_seoul = df[df["Station_ID"].isin(SEOUL_STATIONS)].pivot(index="Datetime", columns="Station_ID", values="winddirection_10m").mean(axis=1)
wind_seoul_test = wind_seoul.reindex(resid_local.index)

TOLERANCE = 45
for LAG in [1, 3, 6, 12, 24]:
    seoul_resid_lag = seoul_resid.shift(LAG)
    downwind_corrs, other_corrs = [], []
    for s in other_stations2:
        angle_diff = np.abs(((wind_seoul_test - (bearings[s] + 180)) + 180) % 360 - 180)
        downwind_mask = angle_diff <= TOLERANCE
        combined_s = pd.concat([resid_local[s].rename("target"), seoul_resid_lag.rename("seoul_lag"),
                                 downwind_mask.rename("downwind")], axis=1).dropna()
        if combined_s["downwind"].sum() > 100:
            c_down = combined_s.loc[combined_s["downwind"], "target"].corr(combined_s.loc[combined_s["downwind"], "seoul_lag"])
            c_other = combined_s.loc[~combined_s["downwind"], "target"].corr(combined_s.loc[~combined_s["downwind"], "seoul_lag"])
            downwind_corrs.append(c_down)
            other_corrs.append(c_other)
    print(f"LAG={LAG}h: mean corr DOWNWIND of Seoul={np.nanmean(downwind_corrs):.4f} (n={len(downwind_corrs)})  "
          f"vs NOT downwind={np.nanmean(other_corrs):.4f}")


25 Seoul stations: [np.int64(111121), np.int64(111123), np.int64(111131), np.int64(111141), np.int64(111142), np.int64(111151), np.int64(111152), np.int64(111161), np.int64(111171), np.int64(111181), np.int64(111191), np.int64(111201), np.int64(111212), np.int64(111221), np.int64(111231), np.int64(111241), np.int64(111251), np.int64(111261), np.int64(111262), np.int64(111273), np.int64(111274), np.int64(111281), np.int64(111291), np.int64(111301), np.int64(111311)]
LAG=1h: mean corr DOWNWIND of Seoul=-0.1011 (n=151)  vs NOT downwind=-0.1277
LAG=3h: mean corr DOWNWIND of Seoul=-0.0714 (n=151)  vs NOT downwind=-0.1052
LAG=6h: mean corr DOWNWIND of Seoul=-0.0409 (n=151)  vs NOT downwind=-0.0760
LAG=12h: mean corr DOWNWIND of Seoul=-0.0116 (n=151)  vs NOT downwind=-0.0431
LAG=24h: mean corr DOWNWIND of Seoul=0.0346 (n=151)  vs NOT downwind=-0.0039


In [16]:
from scipy import stats

LAG = 24
seoul_resid_lag = seoul_resid.shift(LAG)
diffs = []
for s in other_stations2:
    angle_diff = np.abs(((wind_seoul_test - (bearings[s] + 180)) + 180) % 360 - 180)
    downwind_mask = angle_diff <= TOLERANCE
    combined_s = pd.concat([resid_local[s].rename("target"), seoul_resid_lag.rename("seoul_lag"),
                             downwind_mask.rename("downwind")], axis=1).dropna()
    if combined_s["downwind"].sum() > 100 and (~combined_s["downwind"]).sum() > 100:
        c_down = combined_s.loc[combined_s["downwind"], "target"].corr(combined_s.loc[combined_s["downwind"], "seoul_lag"])
        c_other = combined_s.loc[~combined_s["downwind"], "target"].corr(combined_s.loc[~combined_s["downwind"], "seoul_lag"])
        if not (np.isnan(c_down) or np.isnan(c_other)):
            diffs.append(c_down - c_other)

diffs = np.array(diffs)
print(f"n_stations={len(diffs)}  mean(downwind - not_downwind)={diffs.mean():.4f}  std={diffs.std():.4f}")
t_stat, p_value = stats.ttest_1samp(diffs, 0)
print(f"paired t-test vs 0: t={t_stat:.3f}  p={p_value:.5f}")
print(f"fraction of stations where downwind > not_downwind: {(diffs > 0).mean():.1%}")


n_stations=151  mean(downwind - not_downwind)=0.0385  std=0.0665
paired t-test vs 0: t=7.084  p=0.00000
fraction of stations where downwind > not_downwind: 72.2%


In [18]:
val_test_meta = pd.concat([
    val[["Station_ID"]].assign(resid=y_val_arr - pred_val),
    test[["Station_ID"]].assign(resid=y_test_arr - pred_test),
])
resid_wide_vt = val_test_meta.pivot_table(index=val_test_meta.index, columns="Station_ID", values="resid")
resid_national_mean_vt = resid_wide_vt.mean(axis=1)
resid_local_vt = resid_wide_vt.sub(resid_national_mean_vt, axis=0).reindex(columns=station_cols)

wind_dir_wide_vt = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_cols].reindex(resid_local_vt.index)
wind_speed_wide_vt = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_cols].reindex(resid_local_vt.index)

wind_dir_lag_vt = wind_dir_wide_vt.shift(LAG).to_numpy()
wind_speed_lag_vt = wind_speed_wide_vt.shift(LAG).to_numpy()
resid_lag_vt = resid_local_vt.shift(LAG).to_numpy()

T_vt = resid_lag_vt.shape[0]
correction_vt = np.full((T_vt, n), np.nan)
for t in range(T_vt):
    wd, ws, rv = wind_dir_lag_vt[t], wind_speed_lag_vt[t], resid_lag_vt[t]
    if np.isnan(wd).all():
        continue
    angle_diff = np.abs(((wd[None, :] - bearing_from) + 180) % 360 - 180)
    aligned = angle_diff <= TOLERANCE
    w = aligned * decay * ws[None, :]
    w_valid = w * (~np.isnan(rv))[None, :]
    denom = w_valid.sum(axis=1)
    num = np.nansum(w_valid * rv[None, :], axis=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        correction_vt[t] = np.where(denom > 0, num / denom, 0.0)

correction_vt_df = pd.DataFrame(correction_vt, index=resid_local_vt.index, columns=station_cols)
correction_vt_series = correction_vt_df.stack()
correction_vt_series.index.names = ["Datetime", "Station_ID"]

val_multi_idx = pd.MultiIndex.from_arrays([val.index, val["Station_ID"]], names=["Datetime", "Station_ID"])
test_multi_idx = pd.MultiIndex.from_arrays([test.index, test["Station_ID"]], names=["Datetime", "Station_ID"])
correction_val_flat = correction_vt_series.reindex(val_multi_idx).to_numpy()
correction_test_flat = correction_vt_series.reindex(test_multi_idx).to_numpy()

val_valid = ~np.isnan(correction_val_flat)
resid_val_actual = y_val_arr - pred_val
beta, alpha = np.polyfit(correction_val_flat[val_valid], resid_val_actual[val_valid], 1)
print(f"fitted scale (on val only): actual_error = {alpha:.4f} + {beta:.4f} * correction_signal")

test_valid = ~np.isnan(correction_test_flat)
pred_test_corrected2 = pred_test.copy()
pred_test_corrected2[test_valid] = pred_test[test_valid] + alpha + beta * correction_test_flat[test_valid]

rmse_base = np.sqrt(np.mean((pred_test - y_test_arr) ** 2))
rmse_corrected2 = np.sqrt(np.mean((pred_test_corrected2 - y_test_arr) ** 2))
print(f"baseline RMSE: {rmse_base:.4f}")
print(f"properly-scaled corrected RMSE: {rmse_corrected2:.4f}")


fitted scale (on val only): actual_error = -0.4761 + 0.0512 * correction_signal
baseline RMSE: 12.6266
properly-scaled corrected RMSE: 12.5360


In [19]:
val_valid_mask = ~np.isnan(correction_val_flat)
pred_val_corrected2 = pred_val.copy()
pred_val_corrected2[val_valid_mask] = pred_val[val_valid_mask] + alpha + beta * correction_val_flat[val_valid_mask]

vol_val_c = X_val_arr[:, vol_col]
vol_test_c = X_test_arr[:, vol_col]

for name, c in THRESHOLDS.items():
    y_val_label = (val["target"] > c).astype(int).to_numpy()
    y_test_label = (test["target"] > c).astype(int).to_numpy()
    trail_val = np.nan_to_num(val[f"trail_rate_{name}"].to_numpy(), nan=frac_exceeding.mean())
    trail_test = np.nan_to_num(test[f"trail_rate_{name}"].to_numpy(), nan=frac_exceeding.mean())

    Zval_orig = np.column_stack([pred_val, vol_val_c, trail_val])
    S_orig = LogisticRegression(max_iter=1000).fit(Zval_orig, y_val_label)
    p_test_orig = S_orig.predict_proba(np.column_stack([pred_test, vol_test_c, trail_test]))[:, 1]

    Zval_corr = np.column_stack([pred_val_corrected2, vol_val_c, trail_val])
    S_corr = LogisticRegression(max_iter=1000).fit(Zval_corr, y_val_label)
    p_test_corr = S_corr.predict_proba(np.column_stack([pred_test_corrected2, vol_test_c, trail_test]))[:, 1]

    print(f"\n=== {name} (c={c}) ===")
    print(f"AUC:    original={roc_auc_score(y_test_label, p_test_orig):.4f}   wind-corrected={roc_auc_score(y_test_label, p_test_corr):.4f}")
    print(f"PR-AUC: original={average_precision_score(y_test_label, p_test_orig):.4f}   wind-corrected={average_precision_score(y_test_label, p_test_corr):.4f}")
    print(f"Brier:  original={brier_score_loss(y_test_label, p_test_orig):.4f}   wind-corrected={brier_score_loss(y_test_label, p_test_corr):.4f}")



=== bad_36 (c=36.0) ===
AUC:    original=0.8426   wind-corrected=0.8419
PR-AUC: original=0.3849   wind-corrected=0.3845
Brier:  original=0.0704   wind-corrected=0.0705

=== very_bad_76 (c=76.0) ===
AUC:    original=0.8408   wind-corrected=0.8373
PR-AUC: original=0.0502   wind-corrected=0.0523
Brier:  original=0.0073   wind-corrected=0.0075


In [22]:
# ================================================================
# Wind-directed temporal graph model, v2: fixes to the 3 issues
# identified in v1 --
#  1) custom layer doing a TARGET-NORMALIZED weighted MEAN of
#     neighbors (matching the validated closed-form correction),
#     not GCNConv's symmetric spectral normalization (built for
#     undirected graphs, wrong fit for our directed, asymmetric one)
#  2) self-information gets its own dedicated linear pathway,
#     never diluted by however strong the wind happens to be
#  3) one graph-conv layer instead of two (no incidental 2-hop
#     mixing within a single hour's snapshot)
#  5) edge weights scaled by their train-period std so their range
#     is controlled instead of raw, unbounded wind speed
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os, time

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 57.6, 73.6
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index = torch.tensor(np.stack([src_idx, dst_idx]), dtype=torch.long)
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
print(f"candidate graph: {E} directed edges")

WINDOW, HORIZON = 8, 8
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
train_end = int(np.searchsorted(years, 2019))
val_end = int(np.searchsorted(years, 2020))

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy()
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy()

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
edge_weight_by_hour = np.nan_to_num(decay_edge[None, :] * cos_align * speed_src, nan=0.0).astype(np.float32)

train_nonzero = edge_weight_by_hour[:train_end][edge_weight_by_hour[:train_end] > 0]
weight_scale = train_nonzero.std()
print(f"edge weight scale (train, nonzero std): {weight_scale:.4f}")
edge_weight_by_hour = (edge_weight_by_hour / weight_scale).astype(np.float32)
print(f"edge_weight_by_hour: {edge_weight_by_hour.shape}, {edge_weight_by_hour.nbytes/1e9:.2f} GB")
edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)

wdir_sin = np.sin(np.radians(wind_dir_arr))
wdir_cos = np.cos(np.radians(wind_dir_arr))
TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
time_arr = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] + [wind_speed_arr, wdir_sin, wdir_cos], axis=-1)
static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()

time_train = time_arr[:train_end]
t_mean = np.nanmean(time_train, axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_train, axis=(0, 1), keepdims=True) + 1e-6
time_arr = np.nan_to_num((time_arr - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32)

pm25_col_idx = TIME_FEATS_FULL.index("PM25")

def make_windows(start, end):
    X_list, y_list, starts_list = [], [], []
    for t in range(start, end - WINDOW - HORIZON + 1):
        X_list.append(time_arr[t:t + WINDOW])
        y_list.append(time_arr[t + WINDOW + HORIZON - 1, :, pm25_col_idx])
        starts_list.append(t)
    return np.stack(X_list), np.stack(y_list), np.array(starts_list)

X_train, y_train, starts_train = make_windows(0, train_end)
X_val, y_val, starts_val = make_windows(train_end, val_end)
X_test, y_test, starts_test = make_windows(val_end, n_time)
print(f"windows: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}, n_feats={n_feats}")

Xtr_t, ytr_t = torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)
Xva_t, yva_t = torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)
Xte_t, yte_t = torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32)
starts_train_t, starts_val_t, starts_test_t = map(lambda a: torch.tensor(a, dtype=torch.long), (starts_train, starts_val, starts_test))

def add_static(x_time_batch):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def batch_edge_index(edge_index, num_nodes, batch_size):
    return torch.cat([edge_index + i * num_nodes for i in range(batch_size)], dim=1)

def gather_edge_weight_seq(starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    return edge_weight_by_hour_t[idx]

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        return self.lin_self(x) + self.lin_neigh(agg_mean)

class WindTemporalGCN(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.conv1 = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        ei_b = batch_edge_index(edge_index, N, B)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1) if edge_weight_seq is not None else torch.ones(ei_b.shape[1], device=xt.device)
            h = torch.relu(self.conv1(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return self.head(embed).squeeze(-1)

class TemporalOnlyGRU(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index=None, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return self.head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
CKPT_DIR = f"{BASE}/ckpt_wind_temporal_v2"
os.makedirs(CKPT_DIR, exist_ok=True)
MICRO_BATCH, ACCUM_STEPS, MAX_EPOCHS, WEIGHT_DECAY = 16, 4, 8, 1e-4
print(f"device={DEVICE}")

def run_epoch(model, X, y, starts, use_graph, optimizer, train, micro_batch=MICRO_BATCH, accum_steps=ACCUM_STEPS):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = micro_batch * accum_steps
    for start in range(0, n, eff_batch):
        if train:
            optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for micro_start in range(0, len(batch_idx), micro_batch):
            mb_idx = batch_idx[micro_start:micro_start + micro_batch]
            if len(mb_idx) == 0:
                continue
            xb = add_static(X[mb_idx]).to(DEVICE)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_edge_weight_seq(starts[mb_idx]).to(DEVICE) if use_graph else None
            ei = edge_index.to(DEVICE) if use_graph else None
            with torch.set_grad_enabled(train):
                pred = model(xb, ei, ew_seq)
                loss = ((pred - yb) ** 2).mean()
            if train:
                (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx)
            total_n += len(mb_idx)
        if train:
            optimizer.step()
    return total_loss / total_n

def train_model(name, model, use_graph, max_epochs=MAX_EPOCHS):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
    best_val, best_epoch = float("inf"), -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss = run_epoch(model, Xtr_t, ytr_t, starts_train_t, use_graph, opt, train=True)
        val_loss = run_epoch(model, Xva_t, yva_t, starts_val_t, use_graph, opt, train=False)
        print(f"[{name}] epoch {epoch:2d}  train={train_loss:.4f}  val={val_loss:.4f}  ({time.time()-t0:.0f}s)", flush=True)
        if val_loss < best_val:
            best_val, best_epoch = val_loss, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    test_loss = run_epoch(model, Xte_t, yte_t, starts_test_t, use_graph, None, train=False)
    print(f"[{name}] BEST epoch={best_epoch}  val={best_val:.4f}  test={test_loss:.4f}\n", flush=True)
    return {"name": name, "best_epoch": best_epoch, "val_mse": best_val, "test_mse": test_loss}

results = []
results.append(train_model("wind_temporal_gcn_v2", WindTemporalGCN(n_feats), use_graph=True))
results.append(train_model("no_graph", TemporalOnlyGRU(n_feats), use_graph=False))

summary = pd.DataFrame(results)
print(summary.to_string(index=False))


candidate graph: 5874 directed edges
edge weight scale (train, nonzero std): 3.8032
edge_weight_by_hour: (52608, 5874), 1.24 GB
windows: train=26289, val=8745, test=17529, n_feats=20
device=mps
[wind_temporal_gcn_v2] epoch  1  train=0.5411  val=0.5373  (18s)
[wind_temporal_gcn_v2] epoch  2  train=0.4857  val=0.5519  (16s)
[wind_temporal_gcn_v2] epoch  3  train=0.4761  val=0.5070  (16s)
[wind_temporal_gcn_v2] epoch  4  train=0.4708  val=0.4997  (16s)
[wind_temporal_gcn_v2] epoch  5  train=0.4677  val=0.5107  (16s)
[wind_temporal_gcn_v2] epoch  6  train=0.4642  val=0.4831  (16s)
[wind_temporal_gcn_v2] epoch  7  train=0.4623  val=0.4955  (16s)
[wind_temporal_gcn_v2] epoch  8  train=0.4604  val=0.4883  (16s)
[wind_temporal_gcn_v2] BEST epoch=6  val=0.4831  test=0.3528

[no_graph] epoch  1  train=0.5662  val=0.5535  (11s)
[no_graph] epoch  2  train=0.5082  val=0.5365  (11s)
[no_graph] epoch  3  train=0.5007  val=0.5386  (11s)
[no_graph] epoch  4  train=0.4977  val=0.5415  (11s)
[no_graph] e

In [23]:
SEEDS = [0, 1, 2]
all_results = []

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)
    res_wind = train_model(f"wind_temporal_gcn_v2_seed{seed}", WindTemporalGCN(n_feats), use_graph=True)
    res_wind["seed"], res_wind["model"] = seed, "wind_temporal_gcn_v2"
    all_results.append(res_wind)

    torch.manual_seed(seed)
    np.random.seed(seed)
    res_nograph = train_model(f"no_graph_seed{seed}", TemporalOnlyGRU(n_feats), use_graph=False)
    res_nograph["seed"], res_nograph["model"] = seed, "no_graph"
    all_results.append(res_nograph)

results_df = pd.DataFrame(all_results)
print(results_df[["model", "seed", "best_epoch", "val_mse", "test_mse"]].to_string(index=False))

print("\n=== summary across seeds ===")
print(results_df.groupby("model")[["val_mse", "test_mse"]].agg(["mean", "std"]))

pivot_test = results_df.pivot(index="seed", columns="model", values="test_mse")
pivot_test["wind_wins"] = pivot_test["wind_temporal_gcn_v2"] < pivot_test["no_graph"]
pivot_test["pct_improvement"] = 100 * (pivot_test["no_graph"] - pivot_test["wind_temporal_gcn_v2"]) / pivot_test["no_graph"]
print("\n=== per-seed test MSE comparison ===")
print(pivot_test.to_string())


[wind_temporal_gcn_v2_seed0] epoch  1  train=0.5389  val=0.5446  (17s)
[wind_temporal_gcn_v2_seed0] epoch  2  train=0.4881  val=0.5128  (16s)
[wind_temporal_gcn_v2_seed0] epoch  3  train=0.4782  val=0.5156  (16s)
[wind_temporal_gcn_v2_seed0] epoch  4  train=0.4723  val=0.4989  (16s)
[wind_temporal_gcn_v2_seed0] epoch  5  train=0.4682  val=0.4935  (16s)
[wind_temporal_gcn_v2_seed0] epoch  6  train=0.4657  val=0.4966  (16s)
[wind_temporal_gcn_v2_seed0] epoch  7  train=0.4630  val=0.4919  (16s)
[wind_temporal_gcn_v2_seed0] epoch  8  train=0.4609  val=0.4862  (16s)
[wind_temporal_gcn_v2_seed0] BEST epoch=8  val=0.4862  test=0.3549

[no_graph_seed0] epoch  1  train=0.5557  val=0.5601  (10s)
[no_graph_seed0] epoch  2  train=0.5090  val=0.5470  (10s)
[no_graph_seed0] epoch  3  train=0.5020  val=0.5428  (10s)
[no_graph_seed0] epoch  4  train=0.4965  val=0.5417  (10s)
[no_graph_seed0] epoch  5  train=0.4938  val=0.5419  (10s)
[no_graph_seed0] epoch  6  train=0.4897  val=0.5560  (10s)
[no_graph_

In [24]:
# ================================================================
# Same wind-directed temporal graph model (v2 architecture: target-
# normalized weighted-mean aggregation, protected self-pathway,
# single conv layer, standardized edge weights) vs matched no-graph
# ablation -- now at WINDOW=12h, HORIZON=12h. Run across 3 seeds,
# same as the H=8 robustness check, since that's the rigor level
# this comparison has earned.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os, time

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 57.6, 73.6
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index = torch.tensor(np.stack([src_idx, dst_idx]), dtype=torch.long)
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
print(f"candidate graph: {E} directed edges")

WINDOW, HORIZON = 12, 12
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
train_end = int(np.searchsorted(years, 2019))
val_end = int(np.searchsorted(years, 2020))

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy()
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy()

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
edge_weight_by_hour = np.nan_to_num(decay_edge[None, :] * cos_align * speed_src, nan=0.0).astype(np.float32)

train_nonzero = edge_weight_by_hour[:train_end][edge_weight_by_hour[:train_end] > 0]
weight_scale = train_nonzero.std()
print(f"edge weight scale (train, nonzero std): {weight_scale:.4f}")
edge_weight_by_hour = (edge_weight_by_hour / weight_scale).astype(np.float32)
print(f"edge_weight_by_hour: {edge_weight_by_hour.shape}, {edge_weight_by_hour.nbytes/1e9:.2f} GB")
edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)

wdir_sin = np.sin(np.radians(wind_dir_arr))
wdir_cos = np.cos(np.radians(wind_dir_arr))
TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
time_arr = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] + [wind_speed_arr, wdir_sin, wdir_cos], axis=-1)
static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()

time_train = time_arr[:train_end]
t_mean = np.nanmean(time_train, axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_train, axis=(0, 1), keepdims=True) + 1e-6
time_arr = np.nan_to_num((time_arr - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32)

pm25_col_idx = TIME_FEATS_FULL.index("PM25")

def make_windows(start, end):
    X_list, y_list, starts_list = [], [], []
    for t in range(start, end - WINDOW - HORIZON + 1):
        X_list.append(time_arr[t:t + WINDOW])
        y_list.append(time_arr[t + WINDOW + HORIZON - 1, :, pm25_col_idx])
        starts_list.append(t)
    return np.stack(X_list), np.stack(y_list), np.array(starts_list)

X_train, y_train, starts_train = make_windows(0, train_end)
X_val, y_val, starts_val = make_windows(train_end, val_end)
X_test, y_test, starts_test = make_windows(val_end, n_time)
print(f"windows: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}, n_feats={n_feats}")

Xtr_t, ytr_t = torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)
Xva_t, yva_t = torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)
Xte_t, yte_t = torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32)
starts_train_t, starts_val_t, starts_test_t = map(lambda a: torch.tensor(a, dtype=torch.long), (starts_train, starts_val, starts_test))

def add_static(x_time_batch):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def batch_edge_index(edge_index, num_nodes, batch_size):
    return torch.cat([edge_index + i * num_nodes for i in range(batch_size)], dim=1)

def gather_edge_weight_seq(starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    return edge_weight_by_hour_t[idx]

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        return self.lin_self(x) + self.lin_neigh(agg_mean)

class WindTemporalGCN(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.conv1 = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        ei_b = batch_edge_index(edge_index, N, B)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1) if edge_weight_seq is not None else torch.ones(ei_b.shape[1], device=xt.device)
            h = torch.relu(self.conv1(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return self.head(embed).squeeze(-1)

class TemporalOnlyGRU(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index=None, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return self.head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
CKPT_DIR = f"{BASE}/ckpt_wind_temporal_h12"
os.makedirs(CKPT_DIR, exist_ok=True)
MICRO_BATCH, ACCUM_STEPS, MAX_EPOCHS, WEIGHT_DECAY = 16, 4, 8, 1e-4
print(f"device={DEVICE}")

def run_epoch(model, X, y, starts, use_graph, optimizer, train, micro_batch=MICRO_BATCH, accum_steps=ACCUM_STEPS):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = micro_batch * accum_steps
    for start in range(0, n, eff_batch):
        if train:
            optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for micro_start in range(0, len(batch_idx), micro_batch):
            mb_idx = batch_idx[micro_start:micro_start + micro_batch]
            if len(mb_idx) == 0:
                continue
            xb = add_static(X[mb_idx]).to(DEVICE)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_edge_weight_seq(starts[mb_idx]).to(DEVICE) if use_graph else None
            ei = edge_index.to(DEVICE) if use_graph else None
            with torch.set_grad_enabled(train):
                pred = model(xb, ei, ew_seq)
                loss = ((pred - yb) ** 2).mean()
            if train:
                (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx)
            total_n += len(mb_idx)
        if train:
            optimizer.step()
    return total_loss / total_n

def train_model(name, model, use_graph, max_epochs=MAX_EPOCHS):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
    best_val, best_epoch = float("inf"), -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss = run_epoch(model, Xtr_t, ytr_t, starts_train_t, use_graph, opt, train=True)
        val_loss = run_epoch(model, Xva_t, yva_t, starts_val_t, use_graph, opt, train=False)
        print(f"[{name}] epoch {epoch:2d}  train={train_loss:.4f}  val={val_loss:.4f}  ({time.time()-t0:.0f}s)", flush=True)
        if val_loss < best_val:
            best_val, best_epoch = val_loss, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    test_loss = run_epoch(model, Xte_t, yte_t, starts_test_t, use_graph, None, train=False)
    print(f"[{name}] BEST epoch={best_epoch}  val={best_val:.4f}  test={test_loss:.4f}\n", flush=True)
    return {"name": name, "best_epoch": best_epoch, "val_mse": best_val, "test_mse": test_loss}

SEEDS = [0, 1, 2]
all_results = []
for seed in SEEDS:
    torch.manual_seed(seed); np.random.seed(seed)
    res_wind = train_model(f"wind_temporal_gcn_h12_seed{seed}", WindTemporalGCN(n_feats), use_graph=True)
    res_wind["seed"], res_wind["model"] = seed, "wind_temporal_gcn"
    all_results.append(res_wind)

    torch.manual_seed(seed); np.random.seed(seed)
    res_nograph = train_model(f"no_graph_h12_seed{seed}", TemporalOnlyGRU(n_feats), use_graph=False)
    res_nograph["seed"], res_nograph["model"] = seed, "no_graph"
    all_results.append(res_nograph)

results_df = pd.DataFrame(all_results)
print(results_df[["model", "seed", "best_epoch", "val_mse", "test_mse"]].to_string(index=False))
print("\n=== summary across seeds (H=12) ===")
print(results_df.groupby("model")[["val_mse", "test_mse"]].agg(["mean", "std"]))

pivot_test = results_df.pivot(index="seed", columns="model", values="test_mse")
pivot_test["wind_wins"] = pivot_test["wind_temporal_gcn"] < pivot_test["no_graph"]
pivot_test["pct_improvement"] = 100 * (pivot_test["no_graph"] - pivot_test["wind_temporal_gcn"]) / pivot_test["no_graph"]
print("\n=== per-seed test MSE comparison (H=12) ===")
print(pivot_test.to_string())


candidate graph: 5874 directed edges
edge weight scale (train, nonzero std): 3.8032
edge_weight_by_hour: (52608, 5874), 1.24 GB
windows: train=26281, val=8737, test=17521, n_feats=20
device=mps
[wind_temporal_gcn_h12_seed0] epoch  1  train=0.6216  val=0.6401  (23s)
[wind_temporal_gcn_h12_seed0] epoch  2  train=0.5676  val=0.6193  (22s)
[wind_temporal_gcn_h12_seed0] epoch  3  train=0.5550  val=0.6264  (22s)
[wind_temporal_gcn_h12_seed0] epoch  4  train=0.5481  val=0.5925  (22s)
[wind_temporal_gcn_h12_seed0] epoch  5  train=0.5442  val=0.5907  (22s)
[wind_temporal_gcn_h12_seed0] epoch  6  train=0.5398  val=0.5865  (23s)
[wind_temporal_gcn_h12_seed0] epoch  7  train=0.5363  val=0.5991  (22s)
[wind_temporal_gcn_h12_seed0] epoch  8  train=0.5347  val=0.6012  (22s)
[wind_temporal_gcn_h12_seed0] BEST epoch=6  val=0.5865  test=0.4253

[no_graph_h12_seed0] epoch  1  train=0.6358  val=0.6666  (15s)
[no_graph_h12_seed0] epoch  2  train=0.5917  val=0.6417  (14s)
[no_graph_h12_seed0] epoch  3  trai

In [25]:
# ================================================================
# Same wind-directed temporal graph model vs matched no-graph
# ablation -- WINDOW=24h, HORIZON=24h. Same architecture, features,
# training procedure, and 3-seed robustness check as the H=8/H=12
# runs, so this is directly comparable and completes the horizon
# sweep at the original policy-relevant 24h horizon.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os, time

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 57.6, 73.6
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
E = len(src_idx)
edge_index = torch.tensor(np.stack([src_idx, dst_idx]), dtype=torch.long)
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
print(f"candidate graph: {E} directed edges")

WINDOW, HORIZON = 24, 24
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
train_end = int(np.searchsorted(years, 2019))
val_end = int(np.searchsorted(years, 2020))

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy()
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy()

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
edge_weight_by_hour = np.nan_to_num(decay_edge[None, :] * cos_align * speed_src, nan=0.0).astype(np.float32)

train_nonzero = edge_weight_by_hour[:train_end][edge_weight_by_hour[:train_end] > 0]
weight_scale = train_nonzero.std()
print(f"edge weight scale (train, nonzero std): {weight_scale:.4f}")
edge_weight_by_hour = (edge_weight_by_hour / weight_scale).astype(np.float32)
print(f"edge_weight_by_hour: {edge_weight_by_hour.shape}, {edge_weight_by_hour.nbytes/1e9:.2f} GB")
edge_weight_by_hour_t = torch.tensor(edge_weight_by_hour)

wdir_sin = np.sin(np.radians(wind_dir_arr))
wdir_cos = np.cos(np.radians(wind_dir_arr))
TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
time_arr = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] + [wind_speed_arr, wdir_sin, wdir_cos], axis=-1)
static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()

time_train = time_arr[:train_end]
t_mean = np.nanmean(time_train, axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_train, axis=(0, 1), keepdims=True) + 1e-6
time_arr = np.nan_to_num((time_arr - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32)

pm25_col_idx = TIME_FEATS_FULL.index("PM25")

def make_windows(start, end):
    X_list, y_list, starts_list = [], [], []
    for t in range(start, end - WINDOW - HORIZON + 1):
        X_list.append(time_arr[t:t + WINDOW])
        y_list.append(time_arr[t + WINDOW + HORIZON - 1, :, pm25_col_idx])
        starts_list.append(t)
    return np.stack(X_list), np.stack(y_list), np.array(starts_list)

X_train, y_train, starts_train = make_windows(0, train_end)
X_val, y_val, starts_val = make_windows(train_end, val_end)
X_test, y_test, starts_test = make_windows(val_end, n_time)
print(f"windows: train={len(X_train)}, val={len(X_val)}, test={len(X_test)}, n_feats={n_feats}")

Xtr_t, ytr_t = torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)
Xva_t, yva_t = torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)
Xte_t, yte_t = torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32)
starts_train_t, starts_val_t, starts_test_t = map(lambda a: torch.tensor(a, dtype=torch.long), (starts_train, starts_val, starts_test))

def add_static(x_time_batch):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def batch_edge_index(edge_index, num_nodes, batch_size):
    return torch.cat([edge_index + i * num_nodes for i in range(batch_size)], dim=1)

def gather_edge_weight_seq(starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    return edge_weight_by_hour_t[idx]

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        return self.lin_self(x) + self.lin_neigh(agg_mean)

class WindTemporalGCN(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.conv1 = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        ei_b = batch_edge_index(edge_index, N, B)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1) if edge_weight_seq is not None else torch.ones(ei_b.shape[1], device=xt.device)
            h = torch.relu(self.conv1(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return self.head(embed).squeeze(-1)

class TemporalOnlyGRU(nn.Module):
    def __init__(self, in_dim, hidden=32, gru_hidden=32, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, x_window, edge_index=None, edge_weight_seq=None):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            h = torch.relu(self.fc1(x_window[:, w].reshape(B * N, Fin)))
            h = self.drop(h)
            h = torch.relu(self.fc2(h))
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return self.head(embed).squeeze(-1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
CKPT_DIR = f"{BASE}/ckpt_wind_temporal_h24"
os.makedirs(CKPT_DIR, exist_ok=True)
MICRO_BATCH, ACCUM_STEPS, MAX_EPOCHS, WEIGHT_DECAY = 16, 4, 8, 1e-4
print(f"device={DEVICE}")

def run_epoch(model, X, y, starts, use_graph, optimizer, train, micro_batch=MICRO_BATCH, accum_steps=ACCUM_STEPS):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = micro_batch * accum_steps
    for start in range(0, n, eff_batch):
        if train:
            optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for micro_start in range(0, len(batch_idx), micro_batch):
            mb_idx = batch_idx[micro_start:micro_start + micro_batch]
            if len(mb_idx) == 0:
                continue
            xb = add_static(X[mb_idx]).to(DEVICE)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_edge_weight_seq(starts[mb_idx]).to(DEVICE) if use_graph else None
            ei = edge_index.to(DEVICE) if use_graph else None
            with torch.set_grad_enabled(train):
                pred = model(xb, ei, ew_seq)
                loss = ((pred - yb) ** 2).mean()
            if train:
                (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx)
            total_n += len(mb_idx)
        if train:
            optimizer.step()
    return total_loss / total_n

def train_model(name, model, use_graph, max_epochs=MAX_EPOCHS):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
    best_val, best_epoch = float("inf"), -1
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss = run_epoch(model, Xtr_t, ytr_t, starts_train_t, use_graph, opt, train=True)
        val_loss = run_epoch(model, Xva_t, yva_t, starts_val_t, use_graph, opt, train=False)
        print(f"[{name}] epoch {epoch:2d}  train={train_loss:.4f}  val={val_loss:.4f}  ({time.time()-t0:.0f}s)", flush=True)
        if val_loss < best_val:
            best_val, best_epoch = val_loss, epoch
            torch.save(model.state_dict(), ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))
    test_loss = run_epoch(model, Xte_t, yte_t, starts_test_t, use_graph, None, train=False)
    print(f"[{name}] BEST epoch={best_epoch}  val={best_val:.4f}  test={test_loss:.4f}\n", flush=True)
    return {"name": name, "best_epoch": best_epoch, "val_mse": best_val, "test_mse": test_loss}

SEEDS = [0, 1, 2]
all_results = []
for seed in SEEDS:
    torch.manual_seed(seed); np.random.seed(seed)
    res_wind = train_model(f"wind_temporal_gcn_h24_seed{seed}", WindTemporalGCN(n_feats), use_graph=True)
    res_wind["seed"], res_wind["model"] = seed, "wind_temporal_gcn"
    all_results.append(res_wind)

    torch.manual_seed(seed); np.random.seed(seed)
    res_nograph = train_model(f"no_graph_h24_seed{seed}", TemporalOnlyGRU(n_feats), use_graph=False)
    res_nograph["seed"], res_nograph["model"] = seed, "no_graph"
    all_results.append(res_nograph)

results_df = pd.DataFrame(all_results)
print(results_df[["model", "seed", "best_epoch", "val_mse", "test_mse"]].to_string(index=False))
print("\n=== summary across seeds (H=24) ===")
print(results_df.groupby("model")[["val_mse", "test_mse"]].agg(["mean", "std"]))

pivot_test = results_df.pivot(index="seed", columns="model", values="test_mse")
pivot_test["wind_wins"] = pivot_test["wind_temporal_gcn"] < pivot_test["no_graph"]
pivot_test["pct_improvement"] = 100 * (pivot_test["no_graph"] - pivot_test["wind_temporal_gcn"]) / pivot_test["no_graph"]
print("\n=== per-seed test MSE comparison (H=24) ===")
print(pivot_test.to_string())


candidate graph: 5874 directed edges
edge weight scale (train, nonzero std): 3.8032
edge_weight_by_hour: (52608, 5874), 1.24 GB
windows: train=26257, val=8713, test=17497, n_feats=20
device=mps
[wind_temporal_gcn_h24_seed0] epoch  1  train=0.7624  val=0.8301  (57s)
[wind_temporal_gcn_h24_seed0] epoch  2  train=0.7122  val=0.8295  (40s)
[wind_temporal_gcn_h24_seed0] epoch  3  train=0.7016  val=0.8334  (40s)
[wind_temporal_gcn_h24_seed0] epoch  4  train=0.6955  val=0.8395  (40s)
[wind_temporal_gcn_h24_seed0] epoch  5  train=0.6919  val=0.8296  (40s)
[wind_temporal_gcn_h24_seed0] epoch  6  train=0.6862  val=0.8298  (40s)
[wind_temporal_gcn_h24_seed0] epoch  7  train=0.6818  val=0.8466  (40s)
[wind_temporal_gcn_h24_seed0] epoch  8  train=0.6791  val=0.8335  (41s)
[wind_temporal_gcn_h24_seed0] BEST epoch=2  val=0.8295  test=0.5588

[no_graph_h24_seed0] epoch  1  train=0.7669  val=0.8610  (23s)
[no_graph_h24_seed0] epoch  2  train=0.7359  val=0.8385  (23s)
[no_graph_h24_seed0] epoch  3  trai